# 02 — Master Split

Notebook ini digunakan untuk:

1. Membaca manifest final yang telah lolos audit.
2. Membagi Dataset A menjadi train, validation, dan test.
3. Membagi Dataset B menjadi train, validation, dan test.
4. Membentuk split gabungan Dataset A + Dataset B.
5. Menjaga `split_group_id` agar tidak bocor antar-split.
6. Menyediakan full Dataset A dan Dataset B khusus evaluasi cross-dataset.
7. Mendefinisikan protokol eksperimen A-only, B-only, A+B, A→B, dan B→A.
8. Menyimpan master split, metadata, dan signature secara reproducible.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ============================================================
# CELL 1- IMPORT LIBRARY
# ============================================================

from pathlib import Path
from datetime import datetime
from itertools import combinations

import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from IPython.display import display

warnings.filterwarnings("ignore")

print("Library berhasil dimuat.")

Library berhasil dimuat.


In [ ]:
# ============================================================
# CELL 2 — PATH PROYEK DAN KONFIGURASI
# ============================================================

ROOT_DIR = Path("/content/drive/MyDrive/eksperimen_baru")
BASELINE_V2_DIR = ROOT_DIR / "baseline_v2"

CONFIG_DIR = BASELINE_V2_DIR / "config"
MANIFEST_DIR = BASELINE_V2_DIR / "manifests"
SPLIT_DIR = BASELINE_V2_DIR / "splits"
METRICS_DIR = BASELINE_V2_DIR / "metrics"
FIGURES_DIR = BASELINE_V2_DIR / "figures"
LOGS_DIR = BASELINE_V2_DIR / "logs"
DOCUMENTATION_DIR = BASELINE_V2_DIR / "documentation"

CONFIG_PATH = CONFIG_DIR / "config.json"

DATASET_A_GROUPED_PATH = (
    MANIFEST_DIR
    / "dataset_a_grouped_manifest.csv"
)

DATASET_B_GROUPED_PATH = (
    MANIFEST_DIR
    / "dataset_b_grouped_manifest.csv"
)

MANIFEST_SIGNATURE_PATH = (
    MANIFEST_DIR
    / "dataset_manifest_signatures.json"
)

AUDIT_STATUS_PATH = (
    LOGS_DIR
    / "data_audit_status.json"
)

for directory in [
    SPLIT_DIR,
    METRICS_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    DOCUMENTATION_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

required_files = [
    CONFIG_PATH,
    DATASET_A_GROUPED_PATH,
    DATASET_B_GROUPED_PATH,
    MANIFEST_SIGNATURE_PATH,
    AUDIT_STATUS_PATH
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"File tidak ditemukan: {file_path}"
        )
with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:
    CONFIG = json.load(file)

RANDOM_STATE = int(
    CONFIG["random_state"]
)

TARGET_CLASSES = list(
    CONFIG["target_classes"]
)

TRAIN_RATIO = float(
    CONFIG["data_split"]["train_ratio"]
)

VAL_RATIO = float(
    CONFIG["data_split"]["validation_ratio"]
)

TEST_RATIO = float(
    CONFIG["data_split"]["internal_test_ratio"]
)

ratio_total = (
    TRAIN_RATIO
    + VAL_RATIO
    + TEST_RATIO
)

if not np.isclose(
    ratio_total,
    1.0
):
    raise ValueError(
        "Total rasio train, validation, "
        "dan test harus sama dengan 1."
    )

print("Random state :", RANDOM_STATE)
print("Target class :", TARGET_CLASSES)
print("Train ratio  :", TRAIN_RATIO)
print("Val ratio    :", VAL_RATIO)
print("Test ratio   :", TEST_RATIO)

Random state : 42
Target class : ['bacterial_leaf_blight', 'blast', 'tungro']
Train ratio  : 0.8
Val ratio    : 0.1
Test ratio   : 0.1


In [ ]:
# ============================================================
# CELL 3 — MEMERIKSA STATUS AUDIT
# ============================================================

with open(
    AUDIT_STATUS_PATH,
    "r",
    encoding="utf-8"
) as file:
    audit_status_record = json.load(file)

AUDIT_STATUS = audit_status_record.get(
    "audit_status",
    "UNKNOWN"
)

BLOCKING_ISSUE_COUNT = int(
    audit_status_record.get(
        "blocking_issue_count",
        -1
    )
)

print("Status audit   :", AUDIT_STATUS)
print("Blocking issue :", BLOCKING_ISSUE_COUNT)

if AUDIT_STATUS != "PASSED":
    raise RuntimeError(
        "Master split hanya boleh dibuat "
        "setelah FINAL DATA AUDIT = PASSED."
    )

if BLOCKING_ISSUE_COUNT != 0:
    raise RuntimeError(
        "Masih terdapat blocking issue."
    )

print("\nAudit gate: LULUS")

Status audit   : PASSED
Blocking issue : 0

Audit gate: LULUS


In [ ]:
# ============================================================
# CELL 4 — MEMBACA MANIFEST FINAL
# ============================================================

dataset_a_manifest = pd.read_csv(
    DATASET_A_GROUPED_PATH
)

dataset_b_manifest = pd.read_csv(
    DATASET_B_GROUPED_PATH
)

required_columns = [
    "dataset",
    "path",
    "relative_path",
    "label",
    "sha256",
    "phash",
    "split_group_id"
]

for manifest_name, dataframe in [
    ("Dataset A", dataset_a_manifest),
    ("Dataset B", dataset_b_manifest)
]:
    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{manifest_name} kehilangan kolom: "
            f"{missing_columns}"
        )

dataset_a_manifest = (
    dataset_a_manifest
    .sort_values(
        [
            "label",
            "relative_path"
        ]
    )
    .reset_index(drop=True)
)

dataset_b_manifest = (
    dataset_b_manifest
    .sort_values(
        [
            "label",
            "relative_path"
        ]
    )
    .reset_index(drop=True)
)

print("Ukuran manifest final:")
print("Dataset A :", dataset_a_manifest.shape)
print("Dataset B :", dataset_b_manifest.shape)

Ukuran manifest final:
Dataset A : (3276, 34)
Dataset B : (240, 34)


In [ ]:
# ============================================================
# CELL 5 — VALIDASI MANIFEST SEBELUM SPLIT
# ============================================================

def validate_grouped_manifest(
    dataframe,
    dataset_name
):
    print(f"\nValidasi {dataset_name}")
    print("-" * 55)

    path_duplicates = int(
        dataframe["path"]
        .duplicated()
        .sum()
    )

    sha_duplicates = int(
        dataframe["sha256"]
        .duplicated()
        .sum()
    )

    missing_labels = int(
        dataframe["label"]
        .isna()
        .sum()
    )

    missing_groups = int(
        dataframe["split_group_id"]
        .isna()
        .sum()
    )

    available_classes = set(
        dataframe["label"]
        .dropna()
        .unique()
    )

    missing_classes = (
        set(TARGET_CLASSES)
        - available_classes
    )

    group_validation = (
        dataframe
        .groupby("split_group_id")
        .agg(
            dataset_count=(
                "dataset",
                "nunique"
            ),
            label_count=(
                "label",
                "nunique"
            ),
            group_size=(
                "path",
                "size"
            )
        )
        .reset_index()
    )

    cross_dataset_groups = int(
        (
            group_validation[
                "dataset_count"
            ] > 1
        ).sum()
    )

    cross_label_groups = int(
        (
            group_validation[
                "label_count"
            ] > 1
        ).sum()
    )

    print("Jumlah citra       :", len(dataframe))
    print("Kelas tersedia     :", sorted(available_classes))
    print("Kelas hilang       :", sorted(missing_classes))
    print("Path duplikat       :", path_duplicates)
    print("SHA-256 duplikat    :", sha_duplicates)
    print("Label kosong        :", missing_labels)
    print("Group ID kosong     :", missing_groups)
    print("Group lintas dataset:", cross_dataset_groups)
    print("Group lintas label  :", cross_label_groups)

    if path_duplicates > 0:
        raise ValueError(
            f"{dataset_name} masih memiliki path duplikat."
        )

    if sha_duplicates > 0:
        raise ValueError(
            f"{dataset_name} masih memiliki exact duplicate."
        )

    if missing_labels > 0:
        raise ValueError(
            f"{dataset_name} memiliki label kosong."
        )

    if missing_groups > 0:
        raise ValueError(
            f"{dataset_name} memiliki split_group_id kosong."
        )

    if missing_classes:
        raise ValueError(
            f"{dataset_name} kehilangan kelas: "
            f"{missing_classes}"
        )

    if cross_dataset_groups > 0:
        raise ValueError(
            f"{dataset_name} memiliki group lintas dataset."
        )

    if cross_label_groups > 0:
        raise ValueError(
            f"{dataset_name} memiliki group lintas label."
        )


validate_grouped_manifest(
    dataset_a_manifest,
    "Dataset A"
)

validate_grouped_manifest(
    dataset_b_manifest,
    "Dataset B"
)

print("\nValidasi manifest final: LULUS")


Validasi Dataset A
-------------------------------------------------------
Jumlah citra       : 3276
Kelas tersedia     : ['bacterial_leaf_blight', 'blast', 'tungro']
Kelas hilang       : []
Path duplikat       : 0
SHA-256 duplikat    : 0
Label kosong        : 0
Group ID kosong     : 0
Group lintas dataset: 0
Group lintas label  : 0

Validasi Dataset B
-------------------------------------------------------
Jumlah citra       : 240
Kelas tersedia     : ['bacterial_leaf_blight', 'blast', 'tungro']
Kelas hilang       : []
Path duplikat       : 0
SHA-256 duplikat    : 0
Label kosong        : 0
Group ID kosong     : 0
Group lintas dataset: 0
Group lintas label  : 0

Validasi manifest final: LULUS


In [ ]:
# ============================================================
# CELL 6 — DISTRIBUSI DATA SEBELUM SPLIT
# ============================================================

for name, dataframe in {
    "Dataset A": dataset_a_manifest,
    "Dataset B": dataset_b_manifest
}.items():

    distribution = (
        dataframe["label"]
        .value_counts()
        .reindex(
            TARGET_CLASSES,
            fill_value=0
        )
    )

    print(f"\n{name}")
    print("Jumlah citra :", len(dataframe))
    print("Jumlah group :",
          dataframe["split_group_id"].nunique())

    display(
        distribution
        .rename_axis("label")
        .reset_index(name="jumlah")
    )


Dataset A
Jumlah citra : 3276
Jumlah group : 2750


,label,jumlah
0,bacterial_leaf_blight,471
1,blast,1727
2,tungro,1078



Dataset B
Jumlah citra : 240
Jumlah group : 239


,label,jumlah
0,bacterial_leaf_blight,80
1,blast,80
2,tungro,80


In [ ]:
# ============================================================
# CELL 7 — FUNGSI PENILAIAN SPLIT
# ============================================================

def get_class_proportion(
    dataframe
):
    return (
        dataframe[
            "label"
        ]
        .value_counts(
            normalize=True
        )
        .reindex(
            TARGET_CLASSES,
            fill_value=0.0
        )
    )


def evaluate_two_way_split(
    left_dataframe,
    right_dataframe,
    reference_dataframe,
    target_right_ratio
):
    """
    Nilai semakin kecil berarti pembagian
    semakin mendekati target.
    """

    actual_right_ratio = (
        len(right_dataframe)
        / len(reference_dataframe)
    )

    size_error = abs(
        actual_right_ratio
        - target_right_ratio
    )

    reference_distribution = (
        get_class_proportion(
            reference_dataframe
        )
    )

    left_distribution = (
        get_class_proportion(
            left_dataframe
        )
    )

    right_distribution = (
        get_class_proportion(
            right_dataframe
        )
    )

    left_class_error = float(
        (
            left_distribution
            - reference_distribution
        )
        .abs()
        .mean()
    )

    right_class_error = float(
        (
            right_distribution
            - reference_distribution
        )
        .abs()
        .mean()
    )

    class_error = (
        left_class_error
        + right_class_error
    )

    score = (
        5.0 * size_error
        + class_error
    )

    return {
        "score": float(score),
        "size_error": float(size_error),
        "class_error": float(class_error),
        "actual_right_ratio": float(
            actual_right_ratio
        )
    }


def search_best_group_split(
    dataframe,
    n_splits,
    target_right_ratio,
    base_seed,
    attempts=40
):
    """
    Mencari kandidat terbaik dari beberapa seed
    menggunakan StratifiedGroupKFold.
    """

    best_candidate = None
    candidate_records = []

    for attempt in range(attempts):
        candidate_seed = (
            base_seed
            + attempt
        )

        splitter = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=candidate_seed
        )

        split_iterator = splitter.split(
            X=dataframe,
            y=dataframe["label"],
            groups=dataframe[
                "split_group_id"
            ]
        )

        for fold_number, (
            left_indices,
            right_indices
        ) in enumerate(
            split_iterator,
            start=1
        ):
            left_dataframe = (
                dataframe
                .iloc[left_indices]
            )

            right_dataframe = (
                dataframe
                .iloc[right_indices]
            )

            left_classes = set(
                left_dataframe[
                    "label"
                ].unique()
            )

            right_classes = set(
                right_dataframe[
                    "label"
                ].unique()
            )

            if (
                left_classes
                != set(TARGET_CLASSES)
                or right_classes
                != set(TARGET_CLASSES)
            ):
                continue

            evaluation = (
                evaluate_two_way_split(
                    left_dataframe=left_dataframe,
                    right_dataframe=right_dataframe,
                    reference_dataframe=dataframe,
                    target_right_ratio=target_right_ratio
                )
            )

            candidate_record = {
                "seed": candidate_seed,
                "fold": fold_number,
                "left_rows": len(
                    left_dataframe
                ),
                "right_rows": len(
                    right_dataframe
                ),
                **evaluation
            }

            candidate_records.append(
                candidate_record
            )

            if (
                best_candidate is None
                or evaluation["score"]
                < best_candidate["score"]
            ):
                best_candidate = {
                    **candidate_record,
                    "left_indices": (
                        left_indices.copy()
                    ),
                    "right_indices": (
                        right_indices.copy()
                    )
                }

    if best_candidate is None:
        raise RuntimeError(
            "Tidak ditemukan kandidat split "
            "yang memiliki seluruh kelas."
        )

    candidate_report = (
        pd.DataFrame(
            candidate_records
        )
        .sort_values(
            [
                "score",
                "seed",
                "fold"
            ]
        )
        .reset_index(drop=True)
    )

    return (
        best_candidate,
        candidate_report
    )


print(
    "Fungsi pencarian group-aware split "
    "berhasil dibuat."
)

Fungsi pencarian group-aware split berhasil dibuat.


In [ ]:
# ============================================================
# CELL 8 — FUNGSI THREE-WAY GROUP-AWARE SPLIT
# ============================================================

def make_three_way_split(
    dataframe,
    prefix,
    base_seed
):

    temp_ratio = VAL_RATIO + TEST_RATIO

    outer_best, outer_report = (
        search_best_group_split(
            dataframe=dataframe,
            n_splits=int(round(1 / temp_ratio)),
            target_right_ratio=temp_ratio,
            base_seed=base_seed
        )
    )

    train = dataframe.iloc[
        outer_best["left_indices"]
    ].copy()

    temporary = dataframe.iloc[
        outer_best["right_indices"]
    ].copy()


    test_in_temp = TEST_RATIO / temp_ratio

    inner_best, inner_report = (
        search_best_group_split(
            dataframe=temporary,
            n_splits=int(round(1 / test_in_temp)),
            target_right_ratio=test_in_temp,
            base_seed=base_seed + 1000
        )
    )

    val = temporary.iloc[
        inner_best["left_indices"]
    ].copy()

    test = temporary.iloc[
        inner_best["right_indices"]
    ].copy()


    splits = {
        f"train_{prefix}": train,
        f"val_{prefix}": val,
        f"test_{prefix}": test
    }

    for split_name, df in splits.items():

        df["split"] = split_name

        splits[split_name] = (
            df.sort_values(
                ["label", "relative_path"]
            )
            .reset_index(drop=True)
        )


    outer_report["stage"] = "outer"
    inner_report["stage"] = "inner"

    search_report = pd.concat(
        [outer_report, inner_report],
        ignore_index=True
    )

    metadata = {
        "outer_seed": int(outer_best["seed"]),
        "outer_fold": int(outer_best["fold"]),
        "inner_seed": int(inner_best["seed"]),
        "inner_fold": int(inner_best["fold"])
    }

    return splits, search_report, metadata


print("Three-way split helper siap.")

Three-way split helper siap.


In [ ]:
# ============================================================
# CELL 9 — SPLIT DATASET A
# ============================================================

splits_a, search_a, metadata_a = (
    make_three_way_split(
        dataset_a_manifest,
        prefix="a",
        base_seed=RANDOM_STATE
    )
)

train_a = splits_a["train_a"]
val_a = splits_a["val_a"]
test_a = splits_a["test_a"]

search_a["source_dataset"] = "Dataset_A"

print("Dataset A:")
print("Train :", len(train_a))
print("Val   :", len(val_a))
print("Test  :", len(test_a))

Dataset A:
Train : 2620
Val   : 327
Test  : 329


In [ ]:
# ============================================================
# CELL 10 — SPLIT DATASET B
# ============================================================

splits_b, search_b, metadata_b = (
    make_three_way_split(
        dataset_b_manifest,
        prefix="b",
        base_seed=RANDOM_STATE
    )
)

train_b = splits_b["train_b"]
val_b = splits_b["val_b"]
test_b = splits_b["test_b"]

search_b["source_dataset"] = "Dataset_B"

print("Dataset B:")
print("Train :", len(train_b))
print("Val   :", len(val_b))
print("Test  :", len(test_b))

Dataset B:
Train : 192
Val   : 24
Test  : 24


In [ ]:
# ============================================================
# CELL 11 — COMBINED SPLIT DAN EXPERIMENT PLAN
# ============================================================

def combine_split(
    split_a,
    split_b,
    split_name
):

    combined = pd.concat(
        [split_a, split_b],
        ignore_index=True
    )

    combined["split"] = split_name

    return (
        combined
        .sort_values(
            ["dataset", "label", "relative_path"]
        )
        .reset_index(drop=True)
    )


train_ab = combine_split(
    train_a, train_b, "train_ab"
)

val_ab = combine_split(
    val_a, val_b, "val_ab"
)

test_ab = combine_split(
    test_a, test_b, "test_ab"
)


# Full dataset hanya untuk cross-dataset test
external_a_full = dataset_a_manifest.copy()
external_a_full["split"] = "external_a_full"

external_b_full = dataset_b_manifest.copy()
external_b_full["split"] = "external_b_full"


experiment_plan = pd.DataFrame([
    ["E1", "A-only",
     "train_a", "val_a", "test_a"],

    ["E2", "B-only",
     "train_b", "val_b", "test_b"],

    ["E3", "A+B",
     "train_ab", "val_ab", "test_ab"],

    ["E4", "A-to-B",
     "train_a", "val_a", "external_b_full"],

    ["E5", "B-to-A",
     "train_b", "val_b", "external_a_full"]
], columns=[
    "experiment",
    "training_regime",
    "train_split",
    "validation_split",
    "primary_test"
])

display(experiment_plan)

,experiment,training_regime,train_split,validation_split,primary_test
0,E1,A-only,train_a,val_a,test_a
1,E2,B-only,train_b,val_b,test_b
2,E3,A+B,train_ab,val_ab,test_ab
3,E4,A-to-B,train_a,val_a,external_b_full
4,E5,B-to-A,train_b,val_b,external_a_full


In [ ]:
# ============================================================
# CELL 12 — VALIDASI PARTITION DAN LEAKAGE
# ============================================================

def validate_partition(
    source,
    splits,
    name
):

    combined = pd.concat(
        list(splits.values()),
        ignore_index=True
    )

    if set(combined["path"]) != set(source["path"]):
        raise ValueError(
            f"{name}: data hilang/berlebih."
        )

    records = []

    for (name_1, df_1), (
        name_2, df_2
    ) in combinations(
        splits.items(), 2
    ):

        record = {
            "dataset": name,
            "split_1": name_1,
            "split_2": name_2,
            "path_overlap":
                len(set(df_1["path"])
                    & set(df_2["path"])),
            "sha_overlap":
                len(set(df_1["sha256"])
                    & set(df_2["sha256"])),
            "group_overlap":
                len(set(df_1["split_group_id"])
                    & set(df_2["split_group_id"]))
        }

        records.append(record)

    report = pd.DataFrame(records)

    if report[
        ["path_overlap",
         "sha_overlap",
         "group_overlap"]
    ].to_numpy().sum() != 0:

        raise RuntimeError(
            f"Leakage ditemukan pada {name}."
        )

    return report


report_a = validate_partition(
    dataset_a_manifest,
    splits_a,
    "Dataset_A"
)

report_b = validate_partition(
    dataset_b_manifest,
    splits_b,
    "Dataset_B"
)

combined_source = pd.concat(
    [
        dataset_a_manifest,
        dataset_b_manifest
    ],
    ignore_index=True
)

report_ab = validate_partition(
    combined_source,
    {
        "train_ab": train_ab,
        "val_ab": val_ab,
        "test_ab": test_ab
    },
    "Dataset_AB"
)

leakage_report = pd.concat(
    [report_a, report_b, report_ab],
    ignore_index=True
)

display(leakage_report)

print("\nValidasi leakage: LULUS")

,dataset,split_1,split_2,path_overlap,sha_overlap,group_overlap
0,Dataset_A,train_a,val_a,0,0,0
1,Dataset_A,train_a,test_a,0,0,0
2,Dataset_A,val_a,test_a,0,0,0
3,Dataset_B,train_b,val_b,0,0,0
4,Dataset_B,train_b,test_b,0,0,0
5,Dataset_B,val_b,test_b,0,0,0
6,Dataset_AB,train_ab,val_ab,0,0,0
7,Dataset_AB,train_ab,test_ab,0,0,0
8,Dataset_AB,val_ab,test_ab,0,0,0



Validasi leakage: LULUS


In [ ]:
# ============================================================
# CELL 13 — MENYIMPAN MASTER SPLIT
# ============================================================

split_dataframes = {
    "train_a": train_a,
    "val_a": val_a,
    "test_a": test_a,

    "train_b": train_b,
    "val_b": val_b,
    "test_b": test_b,

    "train_ab": train_ab,
    "val_ab": val_ab,
    "test_ab": test_ab,

    "external_a_full": external_a_full,
    "external_b_full": external_b_full
}


split_paths = {}

for split_name, dataframe in (
    split_dataframes.items()
):

    path = (
        SPLIT_DIR
        / f"{split_name}.csv"
    )

    dataframe.to_csv(
        path,
        index=False
    )

    split_paths[split_name] = path


EXPERIMENT_PLAN_PATH = (
    SPLIT_DIR
    / "experiment_plan.csv"
)

experiment_plan.to_csv(
    EXPERIMENT_PLAN_PATH,
    index=False
)

print(
    "Semua master split berhasil disimpan."
)

Semua master split berhasil disimpan.


In [ ]:
# ============================================================
# CELL 14 — MENYIMPAN REPORT DAN METADATA
# ============================================================

summary_records = []

for split_name, dataframe in (
    split_dataframes.items()
):

    summary_records.append({
        "split": split_name,
        "jumlah_citra": len(dataframe),
        "jumlah_group":
            dataframe["split_group_id"].nunique(),
        "jumlah_kelas":
            dataframe["label"].nunique(),
        "jumlah_dataset":
            dataframe["dataset"].nunique()
    })


split_summary = pd.DataFrame(
    summary_records
)


core_splits = {
    key: value
    for key, value
    in split_dataframes.items()
    if not key.startswith("external_")
}

split_class_distribution = (
    pd.concat(
        list(core_splits.values()),
        ignore_index=True
    )
    .groupby(
        ["split", "dataset", "label"]
    )
    .size()
    .reset_index(
        name="jumlah_citra"
    )
)


search_report = pd.concat(
    [search_a, search_b],
    ignore_index=True
)


split_summary.to_csv(
    SPLIT_DIR / "split_summary.csv",
    index=False
)

split_class_distribution.to_csv(
    SPLIT_DIR / "split_class_distribution.csv",
    index=False
)

search_report.to_csv(
    LOGS_DIR / "split_search_candidates.csv",
    index=False
)

leakage_report.to_csv(
    METRICS_DIR / "split_leakage_report.csv",
    index=False
)


selection_metadata = {
    "method":
        "two_stage_stratified_group_k_fold",

    "ratios": {
        "train": TRAIN_RATIO,
        "validation": VAL_RATIO,
        "test": TEST_RATIO
    },

    "dataset_a": metadata_a,
    "dataset_b": metadata_b,

    "created_at":
        datetime.now().isoformat()
}


with open(
    LOGS_DIR
    / "split_selection_metadata.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        selection_metadata,
        file,
        indent=4
    )


display(split_summary)

,split,jumlah_citra,jumlah_group,jumlah_kelas,jumlah_dataset
0,train_a,2620,2200,3,1
1,val_a,327,275,3,1
2,test_a,329,275,3,1
3,train_b,192,191,3,1
4,val_b,24,24,3,1
5,test_b,24,24,3,1
6,train_ab,2812,2391,3,2
7,val_ab,351,299,3,2
8,test_ab,353,299,3,2
9,external_a_full,3276,2750,3,1


In [ ]:
# ============================================================
# CELL 15 — MEMBUAT SPLIT SIGNATURES
# ============================================================

def create_split_signature(
    dataframe
):

    columns = [
        "dataset",
        "split",
        "relative_path",
        "label",
        "sha256",
        "split_group_id"
    ]

    text = (
        dataframe[columns]
        .sort_values(
            ["dataset", "relative_path"]
        )
        .astype(str)
        .agg("|".join, axis=1)
        .str.cat(sep="\n")
    )

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


split_signatures = {
    "signature_version":
        "baseline_v2_master_split_v2",

    "splits": {},

    "created_at":
        datetime.now().isoformat()
}


for split_name, dataframe in (
    split_dataframes.items()
):

    split_signatures[
        "splits"
    ][split_name] = {

        "row_count":
            len(dataframe),

        "signature":
            create_split_signature(
                dataframe
            )
    }


SPLIT_SIGNATURE_PATH = (
    SPLIT_DIR
    / "split_signatures.json"
)

with open(
    SPLIT_SIGNATURE_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        split_signatures,
        file,
        indent=4
    )

print(
    "Split signatures berhasil dibuat."
)

Split signatures berhasil dibuat.


In [ ]:
# ============================================================
# CELL 16 — VERIFIKASI FINAL MASTER SPLIT
# ============================================================

for split_name, file_path in (
    split_paths.items()
):

    if not file_path.exists():
        raise FileNotFoundError(
            file_path
        )

    saved = pd.read_csv(
        file_path
    )

    expected = len(
        split_dataframes[
            split_name
        ]
    )

    if len(saved) != expected:
        raise RuntimeError(
            f"{split_name} tidak konsisten."
        )


if not EXPERIMENT_PLAN_PATH.exists():
    raise FileNotFoundError(
        EXPERIMENT_PLAN_PATH
    )


print("=" * 72)
print("02_MASTER_SPLIT — SELESAI")
print("=" * 72)

print("\nDATASET A")
print("Train :", len(train_a))
print("Val   :", len(val_a))
print("Test  :", len(test_a))

print("\nDATASET B")
print("Train :", len(train_b))
print("Val   :", len(val_b))
print("Test  :", len(test_b))

print("\nCOMBINED A+B")
print("Train :", len(train_ab))
print("Val   :", len(val_ab))
print("Test  :", len(test_ab))

print("\nCROSS-DATASET EXTERNAL")
print(
    "Full A:",
    len(external_a_full)
)

print(
    "Full B:",
    len(external_b_full)
)

print("\nAudit status :", AUDIT_STATUS)
print("Leakage      : 0")
print("STATUS       : LULUS")

02_MASTER_SPLIT — SELESAI

DATASET A
Train : 2620
Val   : 327
Test  : 329

DATASET B
Train : 192
Val   : 24
Test  : 24

COMBINED A+B
Train : 2812
Val   : 351
Test  : 353

CROSS-DATASET EXTERNAL
Full A: 3276
Full B: 240

Audit status : PASSED
Leakage      : 0
STATUS       : LULUS
